# Stage One: Exploration and Validation

Wisconsin Environmental Health Explorer — runs the Stage One pipeline end-to-end
(ingest → clean → spatial join → validate → metrics → static maps) and displays
the resulting tables and plots inline.

> This explorer is a descriptive screening tool. It does not estimate individual
> exposure, diagnose disease, establish causality, rank community worthiness, or
> replace environmental-health expertise and community input. Areas highlighted for
> review reflect the selected public indicators and analytic assumptions, not a
> definitive measure of risk or harm.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import geopandas as gpd
import pandas as pd
from IPython.display import Image, display

from src import clean_indicator, clean_svi, ingest, map_layers, metrics, spatial_join, validate

## 1. Ingest raw sources

Downloads are skipped here if `data/raw/` is already populated from a prior run — see `src/ingest.py` for the source list and `data/metadata/ingest_log.csv` for provenance/checksums of the last download.

In [ ]:
if not (REPO_ROOT / "data" / "raw" / "tl_2022_55_tract.zip").exists():
    ingest.main()
else:
    print("Raw files already present; skipping download. Run src/ingest.py directly to refresh.")

## 2. Clean SVI and environmental indicator

In [ ]:
svi = clean_svi.clean_svi()
svi.to_parquet(clean_svi.OUT_PATH, index=False)

indicator = clean_indicator.clean_indicator()
indicator.to_parquet(clean_indicator.OUT_PATH, index=False)

print(f"SVI rows: {len(svi):,}")
print(f"Indicator rows: {len(indicator):,}")
svi.head()

In [ ]:
indicator.head()

## 3. Tract geometries, DNR points, and the spatial join

In [ ]:
tracts = spatial_join.build_tracts()
tracts.to_parquet(spatial_join.TRACTS_OUT, index=False)

dnr_points = spatial_join.build_dnr_points()
dnr_points.to_parquet(spatial_join.DNR_POINTS_OUT, index=False)

screening = spatial_join.build_screening_view(tracts, svi, indicator)
screening.to_parquet(spatial_join.SCREENING_VIEW_OUT, index=False)

print(f"Tracts: {len(tracts):,}")
print(f"DNR points: {len(dnr_points):,}")
print(f"Screening view rows: {len(screening):,}")
print(f"Tracts flagged for review: {int(screening['screening_flag'].sum())}")
screening.drop(columns="geometry").head()

## 4. Validation

Same assertions enforced in `tests/`, run here directly against the in-memory tables.

In [ ]:
validate.check_unique_geoid(tracts)
validate.check_no_null_geometry(tracts)
validate.check_geoid_format(tracts)
validate.check_wisconsin_fips_prefix(tracts)
validate.check_crs(tracts)
validate.check_join_coverage(screening, tracts)
validate.check_no_unexplained_missing_geometry(screening)
validate.check_valid_svi_range(svi)
validate.check_indicator_year_present(indicator)
validate.check_no_negative_pm25(indicator)
validate.check_screening_flag_logic(screening)
validate.check_no_risk_score_column(screening)
print("All validation checks passed.")

## 5. Descriptive statistics

In [ ]:
metrics.main()

## 6. Static exploratory plots

Choropleth of the Wisconsin-relative PM2.5 percentile, and a scatterplot of PM2.5 vs. overall SVI percentile. Both are descriptive only — see the disclaimer printed on each figure.

In [ ]:
map_layers.plot_choropleth(screening)
display(Image(filename=str(map_layers.CHOROPLETH_OUT)))

In [ ]:
map_layers.plot_scatter(screening)
display(Image(filename=str(map_layers.SCATTER_OUT)))